**Clustering Tipe Pengguna Media Sosial**

**Anggota Kelompok:**

1. Said Hasan Nasrallah (19250671) -Ketua 
2. Tata Dwi Tama Puta (19250729)
3. Shabryan Khaylani (19250936)
4. Hon Zie Kaori (19251001)
5. Sheren Putri Toisuta (19250707)

**Business Under standing**


**a. Latar belakang Pemilihan Dataset:**


Penggunaan media sosial telah menjadi bagian yang tidak terpisahkan dari kehidupan masyarakat modern. Namun, dibalik kemudahannya, penggunaan yang tidak terkontrol sering kali menimbulkan fenomena social media fatigue (kelelahan media sosial) dan menimbulkan dampak psikologis yang serius. Sebagaimana dinyatakan oleh (Putri & Aviani, 2023), "penggunaan media sosial yang intensif dapat memicu peningkatan gejala stres pada individu akibat adanya ketidaksesuaian antara ekspektasi digital dengan kenyataan hidup yang dialami.”

Masalah kesehatan mental seperti kecemasan, depresi, dan stres kini menjadi perhatian serius dalam interaksi digital. Data dari berbagai studi menunjukkan adanya keterkaitan antara durasi penggunaan, dan kondisi emosional pengguna. Namun, setiap individu memiliki pola perilaku yang berbeda-beda, ada yang tetap stabil meski aktif, namun ada juga yang sangat rentan meski durasi penggunaannya singkat.

Dalam konteks Data Science, pendekatan Unsupervised Learning melalui algoritma Clustering memberikan solusi untuk mengelompokkan karakteristik pengguna tanpa memerlukan pelabelan manual. Dengan memanfaatkan dataset "Social Media and Mental Health", penelitian ini bertujuan untuk mengidentifikasi segmen pengguna yang memiliki kerentanan tinggi terhadap gangguan mental. Hal ini sejalan dengan urgensi yang disampaikan oleh (Al Yasin, dkk., 2022), bahwa identifikasi dampak media sosial sangat penting sebagai langkah preventif untuk meminimalisir risiko kesehatan mental dan fisik jangka panjang.  Analisis ini diharapkan mampu memberikan kontribusi dalam memahami pola perilaku digital dan mendukung upaya preventif terhadap dampak negatif penggunaan media sosial secara berlebihan.


**b. Menjelaskan Type dan Informasi Data:**
Dari df.shape:
Dataset memiliki 481 baris (responden) dan 21 kolom (variabel/pertanyaan survei).

Dari df.info():
Dataset memiliki 2 tipe data yaitu:

Object (string/teks): Timestamp, Gender, Relationship Status, Occupation Status, Organizations, Do you use social media, Platforms used, dan Waktu harian (kolom 8) — total 8 kolom
Numerik (int64/float64): Usia (float64) dan kolom 9–20 skala 1–5 (int64) — total 13 kolom


Dari df.describe():
Kolom numerik skala 1–5 memiliki nilai minimum 1 dan maksimum 5. Rata-rata usia responden adalah 26 tahun, dengan usia termuda 13 tahun dan tertua 91 tahun.

Dari df.isnull().sum():
Hanya kolom "Organizations" yang memiliki 30 nilai kosong (missing value). Kolom lainnya tidak memiliki missing value.

Dari df.nunique():
Kolom "Platforms used" memiliki 125 kombinasi unik, artinya setiap responden menggunakan kombinasi platform yang sangat beragam sehingga kolom ini sulit diproses langsung.

**Data Preparation**

**a. Jelaskan tahapan apa saja yang digunakan pada tahap Data
Preparation:**

Ada 4 tahapan yang digunakan:

- Filtering data: hapus baris yang tidak relevan
- Seleksi fitur: buang kolom yang tidak dipakai clustering
- Encoding: ubah teks jadi angka
- Normalisasi: samain skala semua kolom


**b. Jelaskan hasil setiap tahapan yang digunakan**

Berdasarkan kode di notebook, ada 2 tahapan yang digunakan:

Menghapus kolom yang tidak relevan (Drop Columns)
Mengubah teks menjadi angka (Encoding)

**Tahap 1. Menghapus Kolom Tidak Relevan**

Kolom yang dihapus ada 4, yaitu:

- Timestamp: Hanya mencatat waktu pengisian form, tidak mencerminkan perilaku pengguna
- Organizations (kolom 5): Terdapat 30 missing value dan tidak relevan untuk clustering perilaku
- Platforms used (kolom 7): Berisi teks bebas dengan 125 kombinasi unik, tidak bisa langsung diolah
- Do you use social media? (kolom 6)Semua responden menjawab Yes/No, tidak berguna sebagai fitur clustering

Hasil: Dataset berkurang dari 21 kolom → 17 kolom. Kolom yang tersisa adalah data demografi (usia, gender, status) dan seluruh kolom perilaku + mental health skala 1–5 (kolom 8–20).

**Tahap 2. Encoding Kolom Waktu Harian**

Kolom 8 "Waktu harian di sosmed" masih berbentuk teks seperti "More than 5 hours" sehingga tidak bisa langsung diproses oleh algoritma. Kolom ini diubah ke angka menggunakan Ordinal Encoding karena nilainya memiliki urutan dari kecil ke besar.


**1. BUSSINES UNDERSTANDING**

In [1]:
import pandas as pd
file= "smmh.csv"
df=pd.read_csv(file)

Membaca Dataset File CSV

In [2]:
kolom_baru = {
    'Timestamp': 'waktu',
    '1. What is your age?': 'umur',
    '2. Gender': 'kelamin',
    '3. Relationship Status': 'status umum',
    '4. Occupation Status': 'status pekerjaan',
    '5. What type of organizations are you affiliated with?': 'organisasi yang terhubung',
    '6. Do you use social media?': 'apakah kamu pengguna media sosial?',
    '7. What social media platforms do you commonly use?': 'platform media sosial',
    '8. What is the average time you spend on social media every day?': 'rata-rata waktu penggunaan media sosial per hari',
    '9. How often do you find yourself using Social media without a specific purpose?': 'seberapa sering menggunakan sosmed tanpa tujuan tertentu',
    '10. How often do you get distracted by Social media when you are busy doing something?': 'distraksi',
    "11. Do you feel restless if you haven't used Social media in a while?": 'kelelahan',
    '12. On a scale of 1 to 5, how easily distracted are you?' : 'distraksi',
    '13. On a scale of 1 to 5, how much are you bothered by worries?' : 'kekhawatiran',
    '14. Do you find it difficult to concentrate on things?' : 'kesulitan konsentrasi',
    '15. On a scale of 1-5, how often do you compare yourself to other successful people through the use of social media?' : 'perbandingan diri sosmed',
    '16. Following the previous question, how do you feel about these comparisons, generally speaking?' : 'perbandingan diri sosmed',
    '17. How often do you look to seek validation from features of social media?' : 'haus validasi',
    '18. How often do you feel depressed or down?' : 'depresi',
    '19. On a scale of 1 to 5, how frequently does your interest in daily activities fluctuate?' : 'fluctuasi minat',
    '20. On a scale of 1 to 5, how often do you face issues regarding sleep?' : 'masalah tidur',
}

df.rename(columns=kolom_baru, inplace=True)
df.head(2).T

,0,1
waktu,4/18/2022 19:18:47,4/18/2022 19:19:28
umur,21.0,21.0
kelamin,Male,Female
status umum,In a relationship,Single
status pekerjaan,University Student,University Student
organisasi yang terhubung,University,University
apakah kamu pengguna media sosial?,Yes,Yes
platform media sosial,"Facebook, Twitter, Instagram, YouTube, Discord...","Facebook, Twitter, Instagram, YouTube, Discord..."
rata-rata waktu penggunaan media sosial per hari,Between 2 and 3 hours,More than 5 hours
seberapa sering menggunakan sosmed tanpa tujuan tertentu,5,4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 481 entries, 0 to 480
Data columns (total 21 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0   waktu                                                     481 non-null    object 
 1   umur                                                      481 non-null    float64
 2   kelamin                                                   481 non-null    object 
 3   status umum                                               481 non-null    object 
 4   status pekerjaan                                          481 non-null    object 
 5   organisasi yang terhubung                                 451 non-null    object 
 6   apakah kamu pengguna media sosial?                        481 non-null    object 
 7   platform media sosial                                     481 non-null    object 
 8   rata-rata waktu peng

📌 Ukuran Dataset
- 481 baris
- 21 kolom

🧠 Tipe Data
- float64: 1 kolom
- int64: 12 kolom
- object: 8 kolom

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
umur,481.0,26.136590,9.915110,13.0,21.0,22.0,26.0,91.0
seberapa sering menggunakan sosmed tanpa tujuan tertentu,481.0,3.553015,1.096299,1.0,3.0,4.0,4.0,5.0
distraksi,481.0,3.320166,1.328137,1.0,2.0,3.0,4.0,5.0
kelelahan,481.0,2.588358,1.257059,1.0,2.0,2.0,3.0,5.0
distraksi,481.0,3.349272,1.175552,1.0,3.0,3.0,4.0,5.0
kekhawatiran,481.0,3.559252,1.283356,1.0,3.0,4.0,5.0,5.0
kesulitan konsentrasi,481.0,3.245322,1.347105,1.0,2.0,3.0,4.0,5.0
perbandingan diri sosmed,481.0,2.831601,1.407835,1.0,2.0,3.0,4.0,5.0
perbandingan diri sosmed,481.0,2.775468,1.056479,1.0,2.0,3.0,3.0,5.0
haus validasi,481.0,2.455301,1.247739,1.0,1.0,2.0,3.0,5.0


- `count`: 481 baris valid
- `mean`: nilai rata-rata
- `std`: sebaran data
- `min`/`max`: nilai terkecil/terbesar
- `age`: mayoritas 20-an tahun
- pertanyaan 9–20: jawaban cenderung di kisaran 2–4

In [5]:
df.isnull().sum()

waktu                                                        0
umur                                                         0
kelamin                                                      0
status umum                                                  0
status pekerjaan                                             0
organisasi yang terhubung                                   30
apakah kamu pengguna media sosial?                           0
platform media sosial                                        0
rata-rata waktu penggunaan media sosial per hari             0
seberapa sering menggunakan sosmed tanpa tujuan tertentu     0
distraksi                                                    0
kelelahan                                                    0
distraksi                                                    0
kekhawatiran                                                 0
kesulitan konsentrasi                                        0
perbandingan diri sosmed                               

### Pemeriksaan Missing Value (`df.isnull().sum()`)

`df.isnull().sum()` menghitung berapa nilai kosong (`NaN`) di setiap kolom DataFrame.

- `Timestamp`: 0
- `1. What is your age?`: 0
- `2. Gender`: 0
- `3. Relationship Status`: 0
- `4. Occupation Status`: 0
- `5. What type of organizations are you affiliated with?`: 30
- `6. Do you use social media?`: 0
- `7. What social media platforms do you commonly use?`: 0
- `8. What is the average time you spend on social media every day?`: 0
- `9. How often do you find yourself using Social media without a specific purpose?`: 0
- `10. How often do you get distracted by Social media when you are busy doing something?`: 0
- `11. Do you feel restless if you haven't used Social media in a while?`: 0
- `12. On a scale of 1 to 5, how easily distracted are you?`: 0
- `13. On a scale of 1 to 5, how much are you bothered by worries?`: 0
- `14. Do you find it difficult to concentrate on things?`: 0
- `15. On a scale of 1-5, how often do you compare yourself to other successful people through the use of social media?`: 0
- `16. Following the previous question, how do you feel about these comparisons, generally speaking?`: 0
- `17. How often do you look to seek validation from features of social media?`: 0
- `18. How often do you feel depressed or down?`: 0
- `19. On a scale of 1 to 5, how frequently does your interest in daily activities fluctuate?`: 0
- `20. On a scale of 1 to 5, how often do you face issues regarding sleep?`: 0

**Kesimpulan singkat:**
- Hanya kolom `5. What type of organizations are you affiliated with?` yang memiliki missing value.
- Jumlah missing di kolom tersebut: `30`.
- Kolom lain lengkap tanpa nilai kosong.

In [6]:
df.nunique()

waktu                                                       480
umur                                                         46
kelamin                                                       9
status umum                                                   4
status pekerjaan                                              4
organisasi yang terhubung                                    18
apakah kamu pengguna media sosial?                            2
platform media sosial                                       125
rata-rata waktu penggunaan media sosial per hari              6
seberapa sering menggunakan sosmed tanpa tujuan tertentu      5
distraksi                                                     5
kelelahan                                                     5
distraksi                                                     5
kekhawatiran                                                  5
kesulitan konsentrasi                                         5
perbandingan diri sosmed                

### Analisis Jumlah Nilai Unik (`df.nunique()`)

`df.nunique()` menunjukkan berapa banyak nilai berbeda yang ada di setiap kolom.

- `Platforms used`: 125 nilai unik
  - artinya setiap responden memakai kombinasi platform yang sangat beragam
  - kolom ini sulit dipakai langsung untuk clustering
- kolom kategorikal lain seperti `Gender`, `Relationship Status`, dan `Occupation Status` cenderung memiliki nilai unik lebih sedikit
  - sehingga lebih mudah untuk di-encode menjadi angka

Kesimpulan:
- Kolom dengan nilai unik yang sangat banyak biasanya perlu direduksi atau dikelompokkan kembali.
- Untuk analisis clustering, kolom seperti `Platforms used` perlu ditangani khusus agar tidak menambah kompleksitas yang berlebihan.

In [7]:
print(df['2. Gender'].value_counts())


KeyError: '2. Gender'

In [ ]:
df['2. Gender'] = df['2. Gender'].apply(lambda x: x if x in ["Male","Female"] else "other")
print("\n")
print(df['2. Gender'].value_counts())



2. Gender
Female    263
Male      211
other       7
Name: count, dtype: int64


**2. DATA PREPERATION**

### Menghapus Kolom yang Tidak Relevan

Sebelum melakukan clustering, kolom-kolom berikut dihapus karena tidak mendukung analisis:

- `Timestamp`: hanya mencatat waktu pengisian survei, bukan perilaku atau kondisi pengguna.
- `5. What type of organizations are you affiliated with?`: memiliki banyak nilai kosong dan kurang relevan untuk pengelompokan.
- `7. What social media platforms do you commonly use?`: berisi kombinasi platform yang sangat beragam sehingga sulit diproses langsung.
- `6. Do you use social media?`: hampir semua responden menjawab sama, sehingga tidak memberikan variasi informasi.

Setelah penghapusan, `df.info()` dijalankan kembali untuk memastikan struktur data yang tersisa.

In [ ]:
df.drop(columns=["Timestamp","5. What type of organizations are you affiliated with?","7. What social media platforms do you commonly use?","6. Do you use social media?"], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 481 entries, 0 to 480
Data columns (total 17 columns):
 #   Column                                                                                                                Non-Null Count  Dtype  
---  ------                                                                                                                --------------  -----  
 0   1. What is your age?                                                                                                  481 non-null    float64
 1   2. Gender                                                                                                             481 non-null    object 
 2   3. Relationship Status                                                                                                481 non-null    object 
 3   4. Occupation Status                                                                                                  481 non-null    object 
 4   8. What 

**Alasan penghapusan:**

- Timestamp, hanya mencatat waktu pengisian survei, dan tidak berhubungan dengan perilaku atau kondisi mental pengguna

- 5. What type of organizations are you affiliated with?,  memiliki banyak nilai kosong, dan kurang relevan untuk pengelompokan pengguna

- 7. What social media platforms do you commonly use?, berisi kombinasi platform yang sangat beragam, dan sulit diolah langsung dalam analisis clustering

- 6. Do you use social media?, hampir semua responden menjawab sama, dan tidak memberikan nilai yang membedakan klaster

Hasil:
dataset menjadi lebih ringkas
hanya menyisakan fitur yang lebih relevan untuk clustering
df.info() digunakan kembali untuk memastikan perubahan struktur data

In [ ]:
kolom_baru={
    
}

In [ ]:
waktu_penggunaan = {
    'Less than an Hour': 1,
    'Between 1 and 2 hours': 2,
    'Between 2 and 3 hours': 3,
    'Between 3 and 4 hours': 4,
    'Between 4 and 5 hours': 5,
    'More than 5 hours': 6
}
df['8. What is the average time you spend on social media every day?'] = df['8. What is the average time you spend on social media every day?'].map(waktu_penggunaan)
df.head(10)

,1. What is your age?,2. Gender,3. Relationship Status,4. Occupation Status,8. What is the average time you spend on social media every day?,9. How often do you find yourself using Social media without a specific purpose?,10. How often do you get distracted by Social media when you are busy doing something?,11. Do you feel restless if you haven't used Social media in a while?,"12. On a scale of 1 to 5, how easily distracted are you?","13. On a scale of 1 to 5, how much are you bothered by worries?",14. Do you find it difficult to concentrate on things?,"15. On a scale of 1-5, how often do you compare yourself to other successful people through the use of social media?","16. Following the previous question, how do you feel about these comparisons, generally speaking?",17. How often do you look to seek validation from features of social media?,18. How often do you feel depressed or down?,"19. On a scale of 1 to 5, how frequently does your interest in daily activities fluctuate?","20. On a scale of 1 to 5, how often do you face issues regarding sleep?"
0,21.0,Male,In a relationship,University Student,3,5,3,2,5,2,5,2,3,2,5,4,5
1,21.0,Female,Single,University Student,6,4,3,2,4,5,4,5,1,1,5,4,5
2,21.0,Female,Single,University Student,4,3,2,1,2,5,4,3,3,1,4,2,5
3,21.0,Female,Single,University Student,6,4,2,1,3,5,3,5,1,2,4,3,2
4,21.0,Female,Single,University Student,3,3,5,4,4,5,5,3,3,3,4,4,1
5,22.0,Female,Single,University Student,3,4,4,2,3,4,3,4,4,3,3,2,4
6,21.0,Female,Married,University Student,4,4,3,2,2,4,3,5,3,4,5,5,3
7,21.0,Female,In a relationship,University Student,6,5,2,3,3,3,1,1,3,1,5,5,1
8,21.0,Female,In a relationship,University Student,6,5,2,3,3,1,1,1,3,1,5,5,1
9,20.0,Male,Single,University Student,1,1,1,1,1,1,1,1,1,1,1,1,1
